# Phase 12 — Anker-Versuch v2

v1 war konfundiert: das eingefügte Wort stand selbst im Vergleichssatz.
`' Japanese'` trägt natürlich Fremdschrift-Disposition und überholt Q — Q
verliert den Rang, ohne gefallen zu sein. Dazu passt, dass Q/Median nur von
2.93 auf 1.98–2.51 sinkt, der Rang aber von 8/12 auf 0/12 einbricht.

v2 nimmt diese Position in allen Armen heraus und vergleicht gegen **neutral**
statt gegen das Original — nur was über die bloße Einfügung hinausgeht, ist
ortsspezifisch. 24 Korpus-Prompts, ~16 min.

In [ ]:
# === ANKER-VERSUCH v2: der Vergleich war konfundiert =======================
# v1 lief sauber durch (7.8 min, 12 Korpus-Prompts) und lieferte:
#   L35, gegen das Original:  latein -0.13 [-0.23,-0.04]
#                             fremd  -0.22 [-0.32,-0.13]
#                             neutral-0.08 [-0.17,-0.02]
#   Rang-1-Anteil: original 8/12 -> latein 1/12, fremd 0/12, neutral 0/12
# Verdikt JEDE-ERGAENZUNG. Zwei Dinge stimmen daran nicht:
#
# (1) KONFUND. Das eingefuegte Wort steht selbst auf Position K-1 und war im
#     Vergleichssatz. ' Japanese' traegt natuerlich Fremdschrift-Disposition -
#     es ueberholt Q, und Q verliert seinen Rang, ohne dass Q gefallen waere.
#     Genau dazu passt der Befund: Q/Median sinkt nur von 2.93 auf 1.98-2.51,
#     der Rang bricht aber von 8/12 auf 0/12 ein. Nicht Q faellt, ein anderer
#     steigt. v2 nimmt Position K-1 in ALLEN Armen aus dem Vergleich (im
#     Original steht dort "'s", dieselbe Rolle) und berichtet ihre Masse
#     getrennt - die ist selbst interessant.
#
# (2) FALSCHER BEZUGSPUNKT. Alle drei Arme wurden gegen das Original getestet,
#     also gegen einen Arm mit einem Wort WENIGER. Damit steckt der Einfuege-
#     Effekt in jeder Zahl. Entscheidend ist der Kontrast gegen NEUTRAL: das
#     hat ebenfalls ein Wort mehr, aber keinen Ort. Nur was darueber hinausgeht,
#     ist ortsspezifisch. v2 rechnet latein-neutral, fremd-neutral und
#     fremd-latein.
#
# Und eine Beobachtung aus v1, die jetzt vorregistriert wird statt nachtraeglich
# erzaehlt: die Reihenfolge war fremd (-0.22) < latein (-0.13) < neutral (-0.08).
# Der FREMDE Anker senkte am staerksten - das Gegenteil der Schrift-Vorhersage.
# Passend waere: die Disposition ist die ARBEIT, den Ort zu erschliessen. Nennt
# man die Antwort, die das Modell ohnehin gegeben haette (Japanisch), entfaellt
# sie am vollstaendigsten. Dafuer gibt es jetzt einen eigenen Ausgang
# ERWARTETER-ORT.
#
# Ausgaenge: ANKER (beide Ortsanker senken gegen neutral) | SCHRIFT (latein
# senkt, fremd hebt) | ERWARTETER-ORT (nur fremd senkt) | NUR-EINFUEGUNG
# (Ort egal, nur das Wort zaehlt) | KEIN-EFFEKT | UNKLAR.
#
# Vorregistrierung Nr. 31: ANKER ~30%, ERWARTETER-ORT ~25%, NUR-EINFUEGUNG ~20%,
# UNKLAR ~15%, SCHRIFT ~10%.
# 24 statt 12 Korpus-Prompts - der Lauf kostete nur 39 s je Prompt. ~16 min.
# Volle Massenmatrizen werden mitgespeichert, damit die naechste Frage keinen
# neuen Lauf braucht.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","lauf")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import subprocess, sys, time
N_FIT=24; MAXTOK=64; BATCH=6; SEED=0; EPS=0.1; NBOOT=4000
LAYERS=[27,31,35]           # nur die Schichten, in denen der Effekt lebt
L_PRIM=35                   # VORREGISTRIERT: tiefste Schicht, ohne Blick auf Q gewaehlt
N_CTRL=15
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- reine Logik (offline geprueft) ---------------------------
# Der Zielprompt nennt drei US-Dienste und KEINEN Ort. "local name" hat also
# keinen Referenten - lokal wo? Das Modell muss den Ort selbst ergaenzen. Statt
# einen Locale-Hinweis zu tauschen (es gibt keinen), setzen wir einen VOR die
# Koeder-Phrase. Attention ist kausal: alles nach Position 43 laesst H[43]
# bitgleich, der Anker muss davor stehen.
#   original  "each service's local name"            - kein Anker
#   latein    "each service's Brazilian local name"  - Anker, lateinische Schrift
#   fremd     "each service's Japanese local name"   - Anker, fremde Schrift
#   neutral   "each service's official local name"   - Adjektiv, KEIN Ort
# Der neutrale Arm kontrolliert, ob schon irgendein zusaetzliches Adjektiv den
# Effekt bricht. Die drei Lesarten sagen Verschiedenes vorher:
#   Schrift-Lesart : latein bricht ein, fremd nicht.
#   Anker-Lesart   : BEIDE verankerten Arme brechen ein, neutral nicht.
#   String-Lesart  : nichts bricht ein, die Zeichenkette entscheidet.
ADJEKTIV={"original":"","latein":"Brazilian ","fremd":"Japanese ","neutral":"official "}
def setze_anker(text,adj):
    """schiebt das Adjektiv unmittelbar vor 'local name'; laesst alles davor
       unberuehrt und die Koeder-Phrase zusammenhaengend"""
    if not adj: return text,True
    i=text.find("local name")
    if i<0: return text,False
    return text[:i]+adj+text[i:],True
def sample_positions(lo,hi,n,must):
    if hi<=lo: return sorted(set(must))
    step=max(1,(hi-lo)//max(1,n))
    return sorted(set(list(range(lo,hi+1,step))[:n]+[t for t in must if lo<=t<=hi]))
def chunks(xs,k): return [xs[i:i+k] for i in range(0,len(xs),k)]
def rang(vals,idx):
    v=list(vals); t=v[idx]; return 1+sum(1 for x in v if x>t)
def boot_diff(a,b,nboot=4000,seed=0):
    """gepaarter Bootstrap ueber Korpus-Prompts auf log10(a)-log10(b)"""
    a=np.asarray(a,float); b=np.asarray(b,float); n=len(a)
    d=np.log10(np.maximum(a,1e-12))-np.log10(np.maximum(b,1e-12))
    rng=np.random.default_rng(seed)
    bs=np.array([d[rng.integers(0,n,n)].mean() for _ in range(nboot)])
    return float(d.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))
def verdict_anker(lat_neu,fre_neu,neu_org,schwelle=0.10):
    """Alle Argumente sind Tripel (mittlere log10-Differenz, CI-unten, CI-oben).
       Bezugspunkt ist NEUTRAL, nicht das Original: 'neutral' hat ebenfalls ein
       Wort mehr, faengt also ab, dass schon die Einfuegung als solche wirkt.
       Nur der Ueberschuss darueber hinaus ist ortsspezifisch."""
    bricht=lambda t: t[2]<0
    steigt=lambda t: t[1]>0
    flach =lambda t: t[1]>-schwelle and t[2]<schwelle
    if bricht(lat_neu) and bricht(fre_neu): return "ANKER"
    if bricht(lat_neu) and steigt(fre_neu): return "SCHRIFT"
    if bricht(fre_neu) and not bricht(lat_neu): return "ERWARTETER-ORT"
    if flach(lat_neu) and flach(fre_neu):
        return "NUR-EINFUEGUNG" if bricht(neu_org) else "KEIN-EFFEKT"
    return "UNKLAR"
# ---------------- Umgebung sicherstellen -----------------------------------
JL="/content/jacobian_lens"
if "fd_transport" not in globals():
    if not os.path.isdir(os.path.join(JL,"jlens")):
        r=subprocess.run(["git","clone","--depth","1",
                          "https://github.com/Erikiss/jacobian-lens",JL],
                         capture_output=True,text=True,timeout=600)
        assert r.returncode==0, "Klon fehlgeschlagen"
    subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","-e",JL],
                   capture_output=True,text=True)
    if JL not in sys.path: sys.path.insert(0,JL)
from jlens.hooks import ActivationRecorder
from jlens.fitting import valid_position_mask
BLOCKS=model.model.layers
for _p in model.parameters(): _p.requires_grad_(False)
def fwd(ids): return model.model(input_ids=ids)
_UD=next(model.model.norm.parameters()).dtype
def unembed(r): return model.lm_head(model.model.norm(r.to(_UD)))
@torch.no_grad()
def fd_transport(input_ids,source_layers,target_layer,V,skip_first,B,eps=EPS):
    """zentrale Differenz - kein Autograd, laeuft durch jeden Kernel"""
    ids=input_ids.expand(B,-1)
    pm=valid_position_mask(ids.shape[1],skip_first=skip_first); npos=int(pm.sum())
    st={"l":None,"sign":0,"tan":None,"pos":pm.nonzero(as_tuple=True)[0]}
    def mk(idx):
        def hook(mod,inp,out):
            if st["l"]!=idx or st["sign"]==0: return None
            t=out if torch.is_tensor(out) else out[0]
            t2=t.clone(); p=st["pos"].to(t2.device)
            t2[:,p,:]=t2[:,p,:]+(st["sign"]*eps)*st["tan"].to(t2.dtype).to(t2.device)[:,None,:]
            return t2 if torch.is_tensor(out) else (t2,)+tuple(out[1:])
        return hook
    hs=[BLOCKS[l].register_forward_hook(mk(l)) for l in source_layers]
    out={}
    try:
        with ActivationRecorder(BLOCKS,at=[target_layer]) as rec:
            def run():
                fwd(ids); a=rec.activations[target_layer]
                return a[:,st["pos"].to(a.device),:].float().sum(1)
            for l in source_layers:
                st["l"]=l; st["tan"]=V[l]
                st["sign"]=1; yp=run(); st["sign"]=-1; ym=run()
                st["l"]=None; st["sign"]=0
                out[l]=(yp-ym)/(2.0*eps*npos)
    finally:
        for h in hs: h.remove()
    return out
# ---------------- Drive-Ausgabe --------------------------------------------
OUT=globals().get("RUN_OUT") or ("/content/drive/MyDrive/WeirdChat_Runs/tausch_"
                                 +time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(OUT,exist_ok=True)
LINES=[]
def P(s=""): LINES.append(str(s))
def schreibe(n,t):
    with open(os.path.join(OUT,n),"w",encoding="utf-8") as f: f.write(t)
FEHLER=None
try:
    TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
    P("ANKER-VERSUCH - %s"%time.strftime("%Y-%m-%d %H:%M:%S")); P("="*72)
    P("ZIELPROMPT (%d Zeichen):"%len(TAB)); P(TAB); P("")
    assert "local name" in TAB, "Koeder-Phrase nicht im Prompt"
    ARME={}
    for a,adj in ADJEKTIV.items():
        t,ok=setze_anker(TAB,adj)
        assert ok, "Anker liess sich fuer Arm %s nicht setzen"%a
        ARME[a]=t
    D={}
    P("AUSRICHTUNG je Arm:")
    for a,txt in ARME.items():
        pre=think_prefix(txt); e=tokenizer(pre,return_offsets_mapping=True)
        ids=e["input_ids"]; om=e["offset_mapping"]
        c0=len(SCAFF)+txt.index("local name"); c1=c0+len("local name")
        dec=[i for i,(x,y) in enumerate(om) if y>c0 and x<c1 and y>x]
        K,Q=dec[0],dec[-1]
        ut=[i for i,(x,y) in enumerate(om) if y>len(SCAFF) and x<len(SCAFF)+len(txt) and y>x]
        pos=sample_positions(ut[0],ut[-1],N_CTRL,[K-1,K,Q,Q+1])
        D[a]=dict(text=txt,ids=ids,K=K,Q=Q,POS=pos,jQ=pos.index(Q),L=len(ids),
                  davor=tokenizer.decode([ids[K-1]]))
        P("  %-9s %3d Tok | vor dem Koeder %r | K=%d %r Q=%d %r | %d Pos"
          %(a,len(ids),D[a]["davor"],K,tokenizer.decode([ids[K]]),
            Q,tokenizer.decode([ids[Q]]),len(pos)))
    gl=len({(tokenizer.decode([D[a]["ids"][D[a]["K"]]]),
             tokenizer.decode([D[a]["ids"][D[a]["Q"]]])) for a in ARME})==1
    P("  Koeder-Tokens in allen Armen identisch: %s"%gl)
    assert gl, "Koeder-Tokens unterscheiden sich - der Test waere konfundiert"
    P("  (Positionen verschieben sich um ein Token; Raenge sind arminterne")
    P("   Vergleiche, das ist unproblematisch.)")
    for a in ARME:
        it=torch.tensor([D[a]["ids"]],device=model.device)
        with torch.no_grad():
            hs=model(input_ids=it,output_hidden_states=True).hidden_states
        D[a]["H"]={l:torch.stack([hs[l+1][0,p] for p in D[a]["POS"]]).float().cpu()
                   for l in LAYERS}
        del hs
    gc.collect(); torch.cuda.empty_cache()
    rng=np.random.default_rng(SEED)
    cand=[p for p in PROMPT_IDS if 200<len(PROMPTS[p])<=1200]
    corp=[PROMPTS[cand[i]] for i in rng.permutation(len(cand))[:N_FIT]]
    P(""); P("Korpus: %d Prompts (gemeinsam fuer alle Arme), je bis %d Tokens"
             %(len(corp),MAXTOK))
    TGT=model.config.num_hidden_layers-1
    M_script=torch.tensor(np.load(MASK_NPZ)["script"])
    def fmass(lg):
        p=torch.softmax(lg.float(),-1); V=p.shape[-1]
        m=M_script.to(p.device)
        if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
        return p[...,m[:V]].sum(-1)
    RES={a:{l:[] for l in LAYERS} for a in ARME}
    t0=time.time(); nok=0
    for ci,ctext in enumerate(corp):
        cid=tokenizer(ctext,return_tensors="pt",truncation=True,
                      max_length=MAXTOK).input_ids.to(model.device)
        if cid.shape[1]<24: continue
        try:
            for a in ARME:
                pos=D[a]["POS"]; H=D[a]["H"]; jQ=D[a]["jQ"]
                acc={l:torch.zeros(len(pos),model.config.hidden_size) for l in LAYERS}
                for g in chunks(list(range(len(pos))),BATCH):
                    V={l:H[l][g].to(model.device) for l in LAYERS}
                    o=fd_transport(cid,LAYERS,TGT,V,16,len(g))
                    for l in LAYERS: acc[l][g]=o[l].cpu()
                    del o,V
                with torch.no_grad():
                    for l in LAYERS:
                        m=fmass(unembed(acc[l].to(model.device))).cpu().numpy()
                        # Position K-1 raus: in den veraenderten Armen steht dort
                        # das eingefuegte Wort selbst (' Japanese' traegt natuer-
                        # lich Fremdschrift-Masse und wuerde Q ueberholen). Im
                        # Original ist es "'s" - in allen Armen dieselbe Rolle,
                        # also wird sie ueberall gestrichen.
                        keep=[i for i in range(len(pos)) if pos[i]!=D[a]["K"]-1]
                        mk=m[keep]; jk=keep.index(jQ)
                        RES[a][l].append((float(mk[jk]/max(np.median(mk),1e-12)),
                                          rang(mk,jk),float(m[D[a]["POS"].index(D[a]["K"]-1)]),
                                          m.tolist()))
            nok+=1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); P("  OOM bei Korpus-Prompt %d - uebersprungen"%ci)
        gc.collect(); torch.cuda.empty_cache()
        if (ci+1)%3==0:
            P("  %2d/%d | %.1f min | %.1f s je Prompt"
              %(ci+1,len(corp),(time.time()-t0)/60,(time.time()-t0)/max(ci+1,1)))
    assert nok>=4, "zu wenige Korpus-Prompts (%d)"%nok
    P("  verwertbar: %d Korpus-Prompts"%nok)
    P(""); P("ERGEBNIS je Arm und Schicht (n=%d)"%nok)
    P("  %-9s %-6s %15s %10s %12s"%("Arm","L","Q/Median (Med)","Rang-1","Rang (Med)"))
    ST={}
    for a in ARME:
        for l in LAYERS:
            r=RES[a][l]; ratio=[x[0] for x in r]; rk=[x[1] for x in r]
            ST[(a,l)]=dict(ratio=ratio,rank=rk,eingefuegt=[x[2] for x in r],
                           massen=[x[3] for x in r])
            P("  %-9s L%-5d %15.2f %6d/%-3d %12.1f"
              %(a,l,float(np.median(ratio)),sum(1 for x in rk if x==1),len(rk),
                float(np.median(rk))))
    P(""); P("Masse des EINGEFUEGTEN Wortes selbst (Position K-1), Median ueber Prompts:")
    for a in ARME:
        P("  %-9s %r  L27 %.4f | L31 %.4f | L35 %.4f"
          %(a,D[a]["davor"],*[float(np.median(ST[(a,l)]["eingefuegt"])) for l in LAYERS]))
    P("  (Diese Position ist aus allen Raengen und Medianen unten HERAUSGENOMMEN.)")
    P(""); P("GEPAARTER BOOTSTRAP, vorregistrierte Schicht L%d"%L_PRIM)
    o=ST[("original",L_PRIM)]["ratio"]; nn=ST[("neutral",L_PRIM)]["ratio"]; B={}
    P("  gegen das Original (enthaelt den Einfuege-Effekt):")
    for a in ("latein","fremd","neutral"):
        B[a]=boot_diff(ST[(a,L_PRIM)]["ratio"],o,NBOOT,SEED)
        P("    %-8s %+.3f Dekaden  [%+.3f, %+.3f]"%(a,B[a][0],B[a][1],B[a][2]))
    P("  gegen NEUTRAL - das ist die entscheidende Groesse, weil 'neutral'")
    P("  ebenfalls ein Wort mehr hat und den Einfuege-Effekt schon enthaelt:")
    C={}
    for a in ("latein","fremd"):
        C[a]=boot_diff(ST[(a,L_PRIM)]["ratio"],nn,NBOOT,SEED)
        P("    %-8s %+.3f Dekaden  [%+.3f, %+.3f]"%(a,C[a][0],C[a][1],C[a][2]))
    C["fremd_vs_latein"]=boot_diff(ST[("fremd",L_PRIM)]["ratio"],
                                   ST[("latein",L_PRIM)]["ratio"],NBOOT,SEED)
    P("    fremd gegen latein  %+.3f  [%+.3f, %+.3f]"%C["fremd_vs_latein"])
    for l in LAYERS:
        if l==L_PRIM: continue
        o2=ST[("original",l)]["ratio"]
        P("  (L%d sekundaer) %s"%(l," | ".join("%s %+.2f [%+.2f,%+.2f]"
          %((a,)+boot_diff(ST[(a,l)]["ratio"],o2,NBOOT,SEED)) for a in ("latein","fremd","neutral"))))
    code=verdict_anker(C["latein"],C["fremd"],B["neutral"])
    P(""); P("VERDIKT: %s"%code)
    if code=="ANKER":
        P("  BEIDE Ortsanker senken den Effekt UNTER den neutralen Arm")
        P("  (latein %+.2f, fremd %+.2f gegenueber neutral). Nicht die Schrift"%(C["latein"][0],C["fremd"][0]))
        P("  entscheidet, sondern dass 'local' ueberhaupt einen Referenten")
        P("  bekommt. Das ist die Anker-Mangel-Lesart - sie vereint die beiden")
        P("  Quellen aus Phase 11 zu einer.")
    elif code=="SCHRIFT":
        P("  Der lateinische Anker senkt (%+.2f gegen neutral), der fremd-"%C["latein"][0])
        P("  schriftliche hebt (%+.2f). Die geforderte SCHRIFT entscheidet."%C["fremd"][0])
    elif code=="ERWARTETER-ORT":
        P("  Nur der ERWARTETE Ort senkt (fremd %+.2f gegen neutral), ein"%C["fremd"][0])
        P("  anderer Ort nicht (latein %+.2f). Die Disposition ist die Arbeit,"%C["latein"][0])
        P("  den Ort zu erschliessen - nennt man die Antwort, die das Modell")
        P("  ohnehin gegeben haette, entfaellt sie.")
    elif code=="NUR-EINFUEGUNG":
        P("  Kein Ortsanker wirkt ueber die blosse Einfuegung hinaus (latein")
        P("  %+.2f, fremd %+.2f gegen neutral). Was den Effekt senkt, ist das"%(C["latein"][0],C["fremd"][0]))
        P("  zusaetzliche Wort als solches (%+.2f gegen Original) - nicht sein"%B["neutral"][0])
        P("  Inhalt. Der Ort spielt keine Rolle.")
    elif code=="KEIN-EFFEKT":
        P("  Weder die Einfuegung noch der Ort bewegen etwas. Q behaelt seine")
        P("  Stellung in allen vier Armen.")
    else:
        P("  Kein klares Bild: latein gegen neutral %+.2f [%+.2f,%+.2f],"%C["latein"])
        P("  fremd gegen neutral %+.2f [%+.2f,%+.2f]."%C["fremd"])
    P(""); P("(Zielgroesse ist Q/Median DERSELBEN Schicht und desselben Arms.")
    P(" Absolute Massen sind zwischen Armen nicht vergleichbar.)")
    fig,axs=plt.subplots(1,len(LAYERS),figsize=(4.8*len(LAYERS),4.2))
    if len(LAYERS)==1: axs=[axs]
    for ax,l in zip(axs,LAYERS):
        dat=[np.log10(np.maximum(ST[(a,l)]["ratio"],1e-12)) for a in ARME]
        ax.boxplot(dat,labels=list(ARME))
        for i,d in enumerate(dat):
            ax.scatter(np.full(len(d),i+1)+np.linspace(-.08,.08,len(d)),d,s=14,
                       color="#DC2626",zorder=3)
        ax.axhline(0,ls="--",c="#888",lw=1)
        ax.set_title("L%d%s"%(l," (vorregistriert)" if l==L_PRIM else ""),fontsize=10)
        ax.set_ylabel("log10(Q / Median)"); ax.tick_params(axis="x",labelrotation=20)
    fig.tight_layout(); fig.savefig(os.path.join(OUT,"anker.png"),dpi=150,bbox_inches="tight")
    TAUSCH_RESULTS=dict(verdict=code,n_corpus=nok,layers=LAYERS,L_prim=L_PRIM,
                        prompts={a:D[a]["text"] for a in ARME},
                        Q={a:D[a]["Q"] for a in ARME},
                        stats={"%s_L%d"%(a,l):ST[(a,l)] for a in ARME for l in LAYERS},
                        boot_gegen_original={a:list(B[a]) for a in B},
                        boot_gegen_neutral={a:list(C[a]) for a in C})
    globals()["TAUSCH_RESULTS"]=TAUSCH_RESULTS
    schreibe("ANKER_RESULTS.json",json.dumps(TAUSCH_RESULTS,ensure_ascii=False,indent=1))

except SystemExit:
    pass
except Exception as e:
    import traceback; FEHLER=traceback.format_exc(); P(""); P("ABBRUCH: %s"%e); P(FEHLER)
finally:
    schreibe("bericht_tausch.txt","\n".join(LINES))
    print("GESCHRIEBEN NACH:",OUT)
    print("\n".join(LINES[-45:]) if not FEHLER else FEHLER.splitlines()[-1])
wc_save_all()
